# 07 -- LSTM baseline (без классификатора **принес ли человек прибыль в принципе**, лайтовая lstm'ка добренькая)

Минимальная архитектура:

`90 дней × 13 каналов → LSTM → Linear → log1p(GMV_30d)`

Обучаемся на `log1p(target)`: метрика соревнования —- RMSLE, поэтому MSE между `log1p(y_pred)` и `log1p(y_true)` почти буквально совпадает с тем, что хотим минимизировать.


In [1]:
from copy import deepcopy
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "lstm" / "meta.json").exists():
            return candidate
    raise FileNotFoundError("Сначала запустите 06_LSTM_Data_Preparation.ipynb")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

TRAIN_CUTOFFS = META["labeled_cutoffs"][:8]      # до 2025-11-15
VAL_CUTOFF = META["labeled_cutoffs"][8]         # 2025-12-15
HOLDOUT_CUTOFF = META["labeled_cutoffs"][9]     # 2026-01-14
INFERENCE_CUTOFF = META["inference_cutoff"]     # 2026-02-13

BATCH_SIZE = 1024
EPOCHS = 5
LR = 1e-3
HIDDEN_SIZE = 64
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("device:", DEVICE)
print("features:", len(META["features"]), META["features"])


device: mps
features: 13 ['search', 'cat', 'searches', 'search_to_cart', 'search_to_ord', 'cat_to_cart', 'cat_to_ord', 'to_cart', 'to_ord', 'gmv_search', 'gmv_cat', 'gmv', 'active']


## Dataset

`X.npy` открывается через memory map: массивы лежат на диске и читаются по батчам, поэтому все snapshot не загружаются в RAM одновременно.


In [2]:
class SequenceDataset(Dataset):
    def __init__(self, cutoff, with_target=True):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        # На диске float16, для вычислений LSTM переводим конкретный пример в float32.
        x = torch.from_numpy(np.array(self.X[i], dtype=np.float32))
        if self.y is None:
            return x

        # Модель сразу предсказывает log1p(target).
        y_log = torch.tensor(np.log1p(float(self.y[i])), dtype=torch.float32)
        return x, y_log


def make_loader(cutoffs, shuffle=False, with_target=True):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]
    dataset = ConcatDataset([SequenceDataset(c, with_target=with_target) for c in cutoffs])
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,   # самый беспроблемный вариант; на сервере можно поставить 2–4
        pin_memory=torch.cuda.is_available(),
    )


## Модель: одна LSTM и одна линейная голова

In [3]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # h[-1] — скрытое состояние после последнего дня последовательности.
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(1)


## Обучение и RMSLE

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.MSELoss()
    total_loss = 0.0

    for X, y_log in loader:
        X = X.to(DEVICE)
        y_log = y_log.to(DEVICE)

        optimizer.zero_grad()
        pred_log = model(X)
        loss = loss_fn(pred_log, y_log)
        loss.backward()

        # Для RNN это дешевая страховка от взрывающихся градиентов.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(X)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_rmsle(model, loader):
    model.eval()
    squared_error = 0.0
    n = 0

    for X, y_log in loader:
        pred_log = model(X.to(DEVICE)).cpu().clamp_min(0)
        squared_error += ((pred_log - y_log) ** 2).sum().item()
        n += len(X)

    return float(np.sqrt(squared_error / n))


def fit_model(train_cutoffs, val_cutoff=None, epochs=EPOCHS):
    train_loader = make_loader(train_cutoffs, shuffle=True)
    val_loader = make_loader(val_cutoff) if val_cutoff is not None else None

    model = LSTMRegressor(
        input_size=len(META["features"]),
        hidden_size=HIDDEN_SIZE,
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_score = np.inf
    best_epoch = epochs

    for epoch in range(1, epochs + 1):
        train_mse = train_one_epoch(model, train_loader, optimizer)

        if val_loader is None:
            print(f"epoch {epoch}: train MSE(log1p)={train_mse:.5f}")
            continue

        val_rmsle = evaluate_rmsle(model, val_loader)
        print(f"epoch {epoch}: train MSE(log1p)={train_mse:.5f}, val RMSLE={val_rmsle:.5f}")

        if val_rmsle < best_score:
            best_score = val_rmsle
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_epoch, best_score


## Временная валидация

Тренируемся на старых cutoff, выбираем эпоху по `2025-12-15`, затем один раз смотрим holdout `2026-01-14`.


In [5]:
model, best_epoch, val_score = fit_model(TRAIN_CUTOFFS, VAL_CUTOFF)

holdout_loader = make_loader(HOLDOUT_CUTOFF)
holdout_score = evaluate_rmsle(model, holdout_loader)

print(f"best epoch: {best_epoch}")
print(f"validation RMSLE: {val_score:.5f}")
print(f"holdout RMSLE: {holdout_score:.5f}")


epoch 1: train MSE(log1p)=3.25308, val RMSLE=1.78544
epoch 2: train MSE(log1p)=3.11416, val RMSLE=1.78699
epoch 3: train MSE(log1p)=3.10656, val RMSLE=1.78300
epoch 4: train MSE(log1p)=3.10345, val RMSLE=1.78206
epoch 5: train MSE(log1p)=3.10074, val RMSLE=1.78119
best epoch: 5
validation RMSLE: 1.78119
holdout RMSLE: 1.72095


> можно еще эпох добавить, усложнить архитектуру, вообще по красоте д.б. (TODO)

## Финальная модель

После оценки качества переобучаем ту же архитектуру на **всех 10 размеченных cutoff**. Число эпох уже выбрано на validation, поэтому inference-cutoff нигде не участвует в настройке модели.


In [6]:
FINAL_TRAIN_CUTOFFS = META["labeled_cutoffs"]
final_model, _, _ = fit_model(
    FINAL_TRAIN_CUTOFFS,
    val_cutoff=None,
    epochs=best_epoch,
)


epoch 1: train MSE(log1p)=3.23430
epoch 2: train MSE(log1p)=3.10374
epoch 3: train MSE(log1p)=3.09746
epoch 4: train MSE(log1p)=3.09422
epoch 5: train MSE(log1p)=3.09038


## Предсказание и submission

In [7]:
@torch.no_grad()
def predict(model, cutoff):
    dataset = SequenceDataset(cutoff, with_target=False)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model.eval()
    pred_log = []
    for X in loader:
        pred_log.append(model(X.to(DEVICE)).cpu().numpy())

    pred_log = np.concatenate(pred_log)
    pred = np.expm1(np.clip(pred_log, 0, None))
    return np.asarray(dataset.users), pred


user_ids, pred = predict(final_model, INFERENCE_CUTOFF)

# Сохраняем в том же порядке, что sample_submit.csv.
sample = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
pred_by_user = pd.Series(pred, index=user_ids)

submission = sample.copy()
submission["predict"] = submission["user_id"].map(pred_by_user)
assert submission["predict"].notna().all()
submission["predict"] = submission["predict"].clip(lower=0)

out_path = SUBMISSION_DIR / "lstm.csv"
submission.to_csv(out_path, index=False)

print("saved:", out_path)
print("rows:", len(submission))
print("zero predictions:", f"{(submission['predict'] == 0).mean():.2%}")
submission.head()


saved: /Users/pinta/Dev/E-CUP-2026/submissions/lstm.csv
rows: 250000
zero predictions: 0.00%


,user_id,predict
0,2,3.873545
1,7,122.240173
2,15,17.352043
3,18,113.870773
4,23,0.849283
